# Flying a simulated Crazyflie in a notebook

The first cell switches cflib to the **cfsim** simulator. Remove (or comment out) `cfsim.enable(...)` to fly a real Crazyflie with exactly the same code.

On your own computer a live 3D window opens (it may appear behind this window). In an online notebook, use `cfsim.enable(viewer=False)` and look at the pictures from `cfsim.show()`.

In [ ]:
import cfsim
cfsim.enable(world='obstacles')      # remove this line to fly the real drone

In [ ]:
import time

import cflib.crtp
from cflib.crazyflie import Crazyflie
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie
from cflib.positioning.motion_commander import MotionCommander
from cflib.utils.multiranger import Multiranger

URI = 'radio://0/80/2M/E7E7E7E7E7'
cflib.crtp.init_drivers()

## 1. Take off, fly forward, turn, land

In [ ]:
with SyncCrazyflie(URI, cf=Crazyflie(rw_cache='./cache')) as scf:
    scf.cf.supervisor.send_arming_request(True)
    time.sleep(1)
    with MotionCommander(scf, default_height=0.5) as mc:
        mc.forward(1.0)
        mc.turn_left(90)
        mc.forward(0.5)
print('Done')

In [ ]:
cfsim.show()

## 2. Fly forward until a wall is close, then turn

Change `SAFE_DISTANCE` and run the cell again. `cfsim.reset()` puts the drone back at the start.

In [ ]:
SAFE_DISTANCE = 0.5
cfsim.reset()

with SyncCrazyflie(URI, cf=Crazyflie(rw_cache='./cache')) as scf:
    scf.cf.supervisor.send_arming_request(True)
    time.sleep(1)
    with MotionCommander(scf, default_height=0.5) as mc, Multiranger(scf) as ranger:
        end = time.time() + 20
        while time.time() < end:
            if ranger.front is not None and ranger.front < SAFE_DISTANCE:
                mc.turn_left(90)
            else:
                mc.start_forward(0.3)
            time.sleep(0.1)
        mc.stop()
print('Done')

In [ ]:
cfsim.show()